In [1]:
%load_ext autoreload
%autoreload 2

# Evaluation based on HTTP Requests observed ($\S 6.1$)

We load the dataset from the paper, which includes data from 460 unique data recipients across the four instrumentation approaches. We use this information in paragraphs for $\S 6.1$

We first provide an overview of the dataset, where we how many HTTP requests were detected by each technique.
We also show the number of unique data recipients detected by each technique.

### Dataset Overview and Advaita-Exclusive Detection

Summary statistics across all vantage points: total HTTP streams,
unique hosts, unique data recipients, and the set of DRs detected
exclusively by Advaita.

In [1]:
import pandas as pd
from tabulate import tabulate
from analysis_utils.stats import get_dr_sets
from analysis_utils.shared_setup import mark_print

http_requests = pd.read_parquet("../data/http_requests.parquet")
http_requests = http_requests[http_requests["http_stream_type"].str.contains("HTTP")]
inst_approaches = http_requests["vantage_type"].unique().tolist()

overview_df = (
    http_requests
    .groupby("vantage_type")
    .agg(
        stream_types=("http_stream_type", "nunique"),
        http_streams=("http_stream_type", "count"),
        data_recipients=("company", "nunique"),
        apps=("app", "nunique"),
    )
    .reset_index()
    .rename(columns={"vantage_type": "Approach"})
)

mark_print("**Per-approach summary:**")
mark_print(tabulate(overview_df, headers="keys", tablefmt="pipe", showindex=False))

**Per-approach summary:**

| Approach   |   stream_types |   http_streams |   data_recipients |   apps |
|:-----------|---------------:|---------------:|------------------:|-------:|
| Advaita    |              3 |          66782 |               384 |     49 |
| Aosp-Sci   |              3 |          40819 |               209 |     45 |
| Frida      |              3 |          25304 |               249 |     43 |
| Mitm       |              3 |          32908 |               284 |     46 |

In [3]:
dr_sets = get_dr_sets(http_requests.copy(), inst_approaches)

all_drs = set().union(*dr_sets.values())
existing_union = set().union(*(dr_sets[vp] for vp in inst_approaches if vp != "Advaita"))
adv_exclusive = dr_sets["Advaita"] - existing_union
best_existing_count = max(len(dr_sets[vp]) for vp in inst_approaches if vp != "Advaita")
best_existing_tehchnique = [vp for vp in inst_approaches if len(dr_sets[vp]) == best_existing_count]
pct_better = (len(dr_sets["Advaita"]) - best_existing_count) / best_existing_count * 100
vp_counts = ", ".join(
    f"{vp}: {len(dr_sets[vp])}" for vp in sorted(dr_sets, key=lambda v: -len(dr_sets[v]))
)

n_hosts = http_requests["host"].nunique()
n_streams = len(http_requests)
stream_types = ", ".join(sorted(http_requests["http_stream_type"].dropna().unique()))

mark_print(
    f"Across all four approaches, the dataset contains **{n_streams:,}** HTTP streams "
    f"to **{n_hosts:,}** unique hosts ({stream_types}). "
    f"Per-approach DRs detected: {vp_counts}. "
    f"Advaita detects **{len(adv_exclusive)}** of **{len(all_drs)}** data recipients "
    f"**exclusively**, invisible to all three existing approaches, "
    f"Advaita detects {len(dr_sets['Advaita'])} DRs in total, "
    f"**{pct_better:.1f}%** more than the best existing approach **{best_existing_tehchnique[0]} ({best_existing_count})**."
)

Across all four approaches, the dataset contains **165,813** HTTP streams to **2,098** unique hosts (HTTP/1.1, HTTP/2, HTTP/3). Per-approach DRs detected: Advaita: 384, Mitm: 284, Frida: 249, Aosp-Sci: 209. Advaita detects **109** of **460** data recipients **exclusively**, invisible to all three existing approaches, Advaita detects 384 DRs in total, **35.2%** more than the best existing approach **Mitm (284)**.

### DR Detection Across HTTP Stream Types
This section computes the number of data recipients detected by each
instrumentation approach, stratified by HTTP stream type (HTTP/1.1, HTTP/2, HTTP/3).
Parenthetical values indicate DRs detected exclusively by that approach.

In [4]:
from IPython.display import display, Markdown
from tabulate import tabulate

detected = http_requests.dropna(subset=["http_stream_type"]).copy()
inst_approaches = sorted(detected["vantage_type"].unique())
stream_types = sorted(detected["http_stream_type"].unique())

rows = []

for st in stream_types:
    sub = detected[detected["http_stream_type"] == st]
    dr_sets = get_dr_sets(sub, inst_approaches)
    combined = set().union(*dr_sets.values())
    row = {"HTTP Stream Type": st, "Combined": len(combined)}
    for v in inst_approaches:
        others = set().union(*(dr_sets[o] for o in inst_approaches if o != v))
        exclusive = len(dr_sets[v] - others)
        row[v] = f"{len(dr_sets[v])} ({exclusive})"
    rows.append(row)

# Total row
all_dr_sets = get_dr_sets(detected, inst_approaches)
all_combined = set().union(*all_dr_sets.values())
total_row = {"HTTP Stream Type": "Total", "Combined": len(all_combined)}
for v in inst_approaches:
    others = set().union(*(all_dr_sets[o] for o in inst_approaches if o != v))
    exclusive = len(all_dr_sets[v] - others)
    total_row[v] = f"{len(all_dr_sets[v])} ({exclusive})"
rows.append(total_row)

table_df = pd.DataFrame(rows)[["HTTP Stream Type"] + inst_approaches + ["Combined"]]
mark_print("**Data recipients per HTTP stream type.** "
                 "Parenthetical values denote DRs exclusive to that approach.")
# mark_print(tabulate(table_df, headers="keys", tablefmt="pipe", showindex=False))
table_df

**Data recipients per HTTP stream type.** Parenthetical values denote DRs exclusive to that approach.

,HTTP Stream Type,Advaita,Aosp-Sci,Frida,Mitm,Combined
0,HTTP/1.1,246 (57),166 (15),176 (8),193 (24),303
1,HTTP/2,270 (100),105 (9),129 (8),169 (24),317
2,HTTP/3,61 (33),22 (5),16 (4),24 (2),74
3,Total,384 (109),209 (14),249 (15),284 (37),460


### Detection Patterns

The following cell computes UpSet-style detection patterns showing which
combinations of instrumentation approaches detect each DR. This supports
the claim that Advaita-exclusive DRs are not incidental but reflect
systematic blind spots in other approches.

In [5]:
from collections import Counter

pattern_counts = Counter()
for dr in all_combined:
    pattern = tuple(v for v in inst_approaches if dr in all_dr_sets[v])
    pattern_counts[pattern] += 1

pattern_rows = [{"Detection Pattern": " + ".join(p), "DR Count": c}
                for p, c in sorted(pattern_counts.items(), key=lambda x: -x[1])]
mark_print(tabulate(pattern_rows, headers="keys", tablefmt="pipe", showindex=False))

| Detection Pattern                 |   DR Count |
|:----------------------------------|-----------:|
| Advaita + Aosp-Sci + Frida + Mitm |        161 |
| Advaita                           |        109 |
| Advaita + Frida + Mitm            |         41 |
| Mitm                              |         37 |
| Advaita + Mitm                    |         28 |
| Advaita + Aosp-Sci                |         16 |
| Frida                             |         15 |
| Aosp-Sci                          |         14 |
| Advaita + Frida                   |         13 |
| Advaita + Aosp-Sci + Frida        |          9 |
| Frida + Mitm                      |          8 |
| Advaita + Aosp-Sci + Mitm         |          7 |
| Aosp-Sci + Frida + Mitm           |          2 |

### Per-DR Stream Count Disparity

We compare the number of HTTP streams Advaita captures per DR against
the best-performing existing for the same DR. This quantifies the depth
of the observability gap beyond binary detection.

In [6]:
dr_streams = detected.groupby(["company", "vantage_type"]).size().reset_index(name="count")
dr_pivot = dr_streams.pivot_table(
    index="company", columns="vantage_type", values="count", fill_value=0
)

techniques = [v for v in inst_approaches if v != "Advaita"]
dr_pivot["best_existing"] = dr_pivot[techniques].max(axis=1)
dr_pivot["disparity"] = dr_pivot["Advaita"] - dr_pivot["best_existing"]

# Top 10 largest disparities
top = dr_pivot.sort_values("disparity", ascending=False).head(10)
top_rows = []
for dr, row in top.iterrows():
    existing_detail = ", ".join(f"{v}={int(row[v])}" for v in techniques)
    top_rows.append({
        "Data Recipient": dr,
        "Advaita": int(row["Advaita"]),
        "Best Existing": int(row["best_existing"]),
        "Disparity": int(row["disparity"]),
    })
mark_print("**Top 10 DRs by stream count disparity (advaita vs. best existing):**")
mark_print(tabulate(top_rows, headers="keys", tablefmt="pipe", showindex=False))

**Top 10 DRs by stream count disparity (advaita vs. best existing):**

| Data Recipient   |   Advaita |   Best Existing |   Disparity |
|:-----------------|----------:|----------------:|------------:|
| Meta             |      5732 |            1261 |        4471 |
| Google AdMob     |      6809 |            3237 |        3572 |
| Shein            |      4149 |            2122 |        2027 |
| ByteDance        |      2700 |             907 |        1793 |
| Snapchat         |      1599 |               9 |        1590 |
| YouTube          |      2140 |             597 |        1543 |
| Google AdManager |      3326 |            2021 |        1305 |
| Spotify          |      1836 |             608 |        1228 |
| Among Us Chat    |      1014 |             216 |         798 |
| OpenAI           |       836 |             125 |         711 |

## Statistical Testing

We perform Cochran's Q test and Friedman test to measure the effectiveness of each TLS instrumentation technique. We use `scipy` package to calculate ranking data, and perform the actual testing.

Across 460 unique data recipients identified in our dataset, Cochran's Q test on binary detection reveals a statistically significant difference across the four vantage points ($Q$ = 209.88, $<10^{-99}$). A complementary Friedman test on stream counts confirms this finding ($\chi_F^2$ = 254.55, $<10^{-54}$). Both tests yield the same strict ordering: Advaita > Mitm > Frida > Aosp-Sci, with mean detection ranks of Advaita (2.95), Mitm (2.51), Frida (2.36), Aosp-Sci (2.18) and mean stream count ranks of Advaita (3.22), Mitm (2.42), Frida (2.25), Aosp-Sci (2.11). Advaita detects 384 data recipients (83.5\%), compared to Aosp-Sci which detects only 209 (45.4\%).

In [7]:
from analysis_utils.stats import perform_friedman, build_summary_table
from analysis_utils.stats import perform_cochran, safe_log_p


cochran = perform_cochran(http_requests, inst_approaches, dropna_cols=["http_stream_type", "company"])
friedman = perform_friedman(http_requests, inst_approaches, pivot_index=["company"])

summary = build_summary_table(cochran["binary_matrix"], friedman["count_pivot"], inst_approaches)
# Combined summary table (Table 4)
sorted_vps = summary["table"].index.tolist()

binary_mean_ranks = ', '.join(f'{v} ({summary["binary_mean_ranks"][v]:.2f})' for v in sorted_vps)
count_mean_ranks = ', '.join(f'{v} ({summary["count_mean_ranks"][v]:.2f})' for v in sorted_vps)

lines = "Both tests yield the same strict ordering: "
lines += f"{' > '.join(sorted_vps)}, "
lines += f"with mean detection ranks of "
lines += f"{binary_mean_ranks}"
lines += f" and mean stream count ranks of "
lines += f"{count_mean_ranks}."
mark_print(lines)

mark_print("**Table 3: Overall detection comparison.**")
mark_print("**Paper reference:** Section 6.1, para 1")
summary["table"].reset_index().rename(columns={"index": "Approach"})

Both tests yield the same strict ordering: Advaita > Mitm > Frida > Aosp-Sci, with mean detection ranks of Advaita (2.95), Mitm (2.51), Frida (2.36), Aosp-Sci (2.18) and mean stream count ranks of Advaita (3.22), Mitm (2.42), Frida (2.25), Aosp-Sci (2.11).

**Table 3: Overall detection comparison.**

**Paper reference:** Section 6.1, para 1

,Approach,DRs Detected,Coverage (%),Mean Rank (Q),Mean Rank (F)
0,Advaita,384/460,83.5,2.95,3.22
1,Mitm,284/460,61.7,2.51,2.42
2,Frida,249/460,54.1,2.36,2.25
3,Aosp-Sci,209/460,45.4,2.18,2.11


### Cochran's Q Test: Binary DR Detection

Cochran's Q tests whether the proportion of data recipients detected
differs across the four instrumentation approaches. Post-hoc pairwise
McNemar tests with Bonferroni correction identify which pairs differ.

**Paper reference:** Section 6.1, para 1

In [8]:
# --- Cell: Cochran's Q + McNemar post-hoc ---
from analysis_utils.stats import perform_cochran, perform_mcnemar_pairwise, safe_log_p

cochran = perform_cochran(http_requests, inst_approaches, dropna_cols=["http_stream_type", "company"])

mark_print(
    f"**Cochran's Q test (binary detection)**\n\n"
    f"$H_0$: {cochran['hypothesis']}\n\n"
    f"$Q$ = {cochran['Q']:.2f}, df = {cochran['df']}, "
    f"*p* = {safe_log_p(cochran['p'])}, "
    f"Kendall's W = {cochran['W']:.4f}, "
    f"*N* = {cochran['N']}.\n\n"
    f"**{cochran['verdict']}:** {cochran['consequence']}"
)

**Cochran's Q test (binary detection)**

$H_0$: The proportion of data recipients detected is equal across all instrumentation approaches.

$Q$ = 209.88, df = 3, *p* = $<10^{-44}$, Kendall's W = 0.1521, *N* = 460.

**Reject $H_0$:** The four approaches differ significantly in which data recipients they detect. At least one approach observes a structurally different subset of DRs.

### McNemar pairwise matrix (Table 4)

**Paper reference:** Section 6.1, para 1

In [13]:
mcnemar = perform_mcnemar_pairwise(cochran["binary_matrix"], inst_approaches)
pairs_rows = []
for r in mcnemar["results"]:
    a, b = r["pair"]
    sig = "***" if r["significant"] else "n.s."
    pairs_rows.append({
        "Pair": f"{a} vs. {b}",
        "A detects, B misses": r["a_not_b"],
        "B detects, A misses": r["b_not_a"],
        "p (corrected)": f"{r['p_corrected']:.2e}" if r["p_corrected"] < 0.01 else f"{r['p_corrected']:.3f}",
        "": sig,
    })

mark_print(
    f"**Pairwise McNemar tests** "
    f"({mcnemar['n_comparisons']} comparisons, "
    f'Bonferroni-adjusted p, *** adjusted $p < 0.05$'
    # f"Bonferroni-corrected $\\alpha$ = {mcnemar['corrected_alpha']:.4f}):"
)
mark_print(tabulate(pairs_rows, headers="keys", tablefmt="pipe", showindex=False))

**Pairwise McNemar tests** (6 comparisons, Bonferroni-adjusted p, *** adjusted $p < 0.05$

| Pair                 |   A detects, B misses |   B detects, A misses |   p (corrected) |     |
|:---------------------|----------------------:|----------------------:|----------------:|:----|
| Advaita vs. Aosp-Sci |                   191 |                    16 |        6.83e-33 | *** |
| Advaita vs. Frida    |                   160 |                    25 |        4.04e-22 | *** |
| Advaita vs. Mitm     |                   147 |                    47 |        7.08e-12 | *** |
| Aosp-Sci vs. Frida   |                    37 |                    77 |        0.00156  | *** |
| Aosp-Sci vs. Mitm    |                    39 |                   114 |        1.32e-08 | *** |
| Frida vs. Mitm       |                    37 |                    72 |        0.00677  | *** |

### Friedman Test: HTTP Request Counts

The Friedman test determines whether the median number of HTTP requests
per data recipient differs across approaches. Nemenyi post-hoc
comparisons identify significant pairwise differences.

**Paper reference:** Section 6.1, para 1


In [10]:
# --- Cell: Friedman + Nemenyi + summary table ---
from analysis_utils.stats import perform_friedman

friedman = perform_friedman(http_requests, inst_approaches, pivot_index=["company"])

mark_print(
    f"**Friedman test (stream counts)**\n\n"
    f"$H_0$: {friedman['hypothesis']}\n\n"
    f"$\\chi^2_F$ = {friedman['chi2']:.2f}, "
    f"*p* = {safe_log_p(friedman['p'])}.\n\n"
    f"**{friedman['verdict']}:** {friedman['consequence']}"
)

mark_print("**Nemenyi post-hoc pairwise comparisons:**")
# nemenyi_fmt = format_nemenyi_upper(friedman["nemenyi"])
friedman["nemenyi"]

**Friedman test (stream counts)**

$H_0$: The median count per company does not differ across instrumentation approaches.

$\chi^2_F$ = 254.55, *p* = $<10^{-54}$.

**Reject $H_0$:** The median count per company differs significantly across approaches. Approaches that detect more subjects also capture more observations per subject, indicating deeper observability rather than mere breadth.

**Nemenyi post-hoc pairwise comparisons:**

vantage_type,Advaita,Aosp-Sci,Frida,Mitm
vantage_type,,,,
Advaita,1.0,0.000000,0.000000,0.000000
Aosp-Sci,0.0,1.000000,0.338172,0.001346
Frida,0.0,0.338172,1.000000,0.186155
Mitm,0.0,0.001346,0.186155,1.000000


Post-hoc pairwise McNemar tests with Bonferroni correction ($\alpha = 0.0083$) confirm that all pairwise differences are statistically significant ($p < 0.01$), establishing a strict total ordering.

## Protocol Stratified Detection Rates

**Paper reference:** $\S 6.1$, para 2

We stratify detection rates by HTTP stream type to examine whether the observability gap varies with transport protocol. Within each protocol version, Cochran's Q confirms significant differences across vantage points (all $p < 0.001$). Reading down the columns reveals a critical asymmetry: Advaita's detection rate is statistically independent of HTTP stream type ($\chi^2$ $p = 0.410$), remaining stable at $81.2\%$ / $85.2\%$ / $82.7\%$ across HTTP/1.1, HTTP/2, HTTP/3 respectively. In contrast, all three existing tools exhibit a significant detection gradient ($p < 0.001$). This confirms that Advaita is the only vantage point whose detection capability is independent of the underlying transport protocol, while existing tools systematically undercount data recipients communicating over modern protocol versions.


### Within-Category Analysis: Cochran's Q per HTTP Version (Table 5)

For each HTTP stream type, Cochran's Q tests whether the four
approaches differ in their detection rate for that protocol.

**Paper reference:** Section 6.1, para 2

In [11]:
from analysis_utils.stats import perform_within_category, safe_log_p

clean = http_requests.dropna(subset=["http_stream_type", "company"])
within_http = perform_within_category(
    http_requests, inst_approaches,
    stratify_col="http_stream_type",
    subject_cols="company",
)

table_rows = []
for w in within_http:
    rates = {
        vp: w["binary_matrix"][vp].sum() / w["N"] * 100
        for vp in inst_approaches
    }
    row = {"Protocol": f"{w['category']} (n={w['N']})"}
    for vp in inst_approaches:
        row[vp] = f"{rates[vp]:.1f}%"
    row["Q"] = f"{w['Q']:.1f}"
    row["p"] = safe_log_p(w["p"]) if w["p"] < 0.001 else f"{w['p']:.3f}"
    table_rows.append(row)

mark_print("**Table 5: Within-category detection rates.**")
mark_print(tabulate(table_rows, headers="keys", tablefmt="pipe", showindex=False))

for w in within_http:
    mark_print(
        f"**{w['category']}** (n={w['N']}): "
        f"$H_0$: {w['hypothesis']} "
        f"$Q$ = {w['Q']:.2f}, *p* = {safe_log_p(w['p'])}. "
        f"**{w['verdict']}:** {w['consequence']}"
    )

**Table 5: Within-category detection rates.**

| Protocol         | Advaita   | Aosp-Sci   | Frida   | Mitm   |     Q | p           |
|:-----------------|:----------|:-----------|:--------|:-------|------:|:------------|
| HTTP/1.1 (n=304) | 81.2%     | 54.6%      | 57.9%   | 63.8%  |  79.6 | $<10^{-16}$ |
| HTTP/2 (n=318)   | 85.2%     | 33.3%      | 40.9%   | 53.5%  | 231.8 | $<10^{-49}$ |
| HTTP/3 (n=75)    | 82.7%     | 29.3%      | 21.3%   | 32.0%  |  71.1 | $<10^{-14}$ |

**HTTP/1.1** (n=304): $H_0$: Detection rates do not differ across approaches for HTTP/1.1. $Q$ = 79.55, *p* = $<10^{-16}$. **Reject $H_0$:** Approaches differ significantly in HTTP/1.1 detection. The observability gap persists within this category.

**HTTP/2** (n=318): $H_0$: Detection rates do not differ across approaches for HTTP/2. $Q$ = 231.76, *p* = $<10^{-49}$. **Reject $H_0$:** Approaches differ significantly in HTTP/2 detection. The observability gap persists within this category.

**HTTP/3** (n=75): $H_0$: Detection rates do not differ across approaches for HTTP/3. $Q$ = 71.14, *p* = $<10^{-14}$. **Reject $H_0$:** Approaches differ significantly in HTTP/3 detection. The observability gap persists within this category.

### Between-Category Analysis: Chi-Squared per Approach

For each approach, a chi-squared test of independence determines
whether its detection rate varies across HTTP stream types. An approach
whose detection is independent of protocol version provides uniform
coverage; one whose detection depends on protocol is structurally
blind to certain protocols.

**Paper reference:** Section 6.1, para 2

In [12]:
# --- Cell: Between-category ---
from analysis_utils.stats import perform_between_category

between = perform_between_category(clean, inst_approaches, stratify_col="http_stream_type")

for vp in inst_approaches:
    b = between[vp]
    if b is None:
        mark_print(f"**{vp}:** Degenerate contingency table (N/A).")
        continue
    min_ok = "OK" if b["min_expected"] >= 5 else "WARNING: < 5"
    lines = f"**{vp}:** $H_0$: {b['hypothesis']} "
    lines += f"$\\chi^2$ = {b['chi2']:.2f}, df = {b['dof']}, "
    p_vals = safe_log_p(b['p']) if b['p'] < 0.001 else f'{b["p"]:.3f}'
    lines += f"*p* = {p_vals}. "
    lines += f"Min expected count: {b['min_expected']:.1f} ({min_ok}). "
    lines += f"**{b['verdict']}:** {b['consequence']}"
    mark_print(lines)

**Advaita:** $H_0$: Advaita's detection rate is independent of http_stream_type. $\chi^2$ = 1.79, df = 2, *p* = 0.410. Min expected count: 12.5 (OK). **Fail to reject $H_0$:** No significant dependence of Advaita's detection on http_stream_type at alpha = 0.05. This approach provides uniform coverage across categories.

**Aosp-Sci:** $H_0$: Aosp-Sci's detection rate is independent of http_stream_type. $\chi^2$ = 35.10, df = 2, *p* = $<10^{-7}$. Min expected count: 31.2 (OK). **Reject $H_0$:** Aosp-Sci's detection depends on http_stream_type.

**Frida:** $H_0$: Frida's detection rate is independent of http_stream_type. $\chi^2$ = 39.07, df = 2, *p* = $<10^{-8}$. Min expected count: 34.2 (OK). **Reject $H_0$:** Frida's detection depends on http_stream_type.

**Mitm:** $H_0$: Mitm's detection rate is independent of http_stream_type. $\chi^2$ = 24.81, df = 2, *p* = $<10^{-5}$. Min expected count: 32.8 (OK). **Reject $H_0$:** Mitm's detection depends on http_stream_type.